In [ ]:
import neurokit2 as nk
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm import tqdm

In [ ]:
def process_research_id(file_path):
    process_research_id = os.listdir(file_path[0])
    split_data = [d.split('.')[0] for d in process_research_id]
    return split_data

In [ ]:
mode = 1 ##
'''
mode = 0 --> non MACE
mode = 1 --> MACE
'''
save_time = '300sec' ##
# resample_rates = [1200, 360, 100] ##
resample_rate = 10000 ##
freq_cut = [500, 150, 40]
non_mace_test_id ='MACE317'
mace_test_id = 'MACE332'


In [ ]:
fc500_file_path = r'V:\dunwei\MACE\dataset\ECG_signal\ch1\sr10000_0.5_500.150.40_ECG_signal\sr10000_0.5_500_ECG_signal/' ##
fc150_file_path = r'V:\dunwei\MACE\dataset\ECG_signal\ch1\sr10000_0.5_500.150.40_ECG_signal\sr10000_0.5_150_ECG_signal/' ##
fc40_file_path = r'V:\dunwei\MACE\dataset\ECG_signal\ch1\sr10000_0.5_500.150.40_ECG_signal\sr10000_0.5_40_ECG_signal/' ##

if mode == 0:
    file_path = [os.path.join(fc500_file_path, 'non_mace'), os.path.join(fc150_file_path, 'non_mace'), os.path.join(fc40_file_path, 'non_mace')]
    save_path = r'V:\dunwei\MACE\dataset\ECG_signal\ch1\sr10000_0.5_500.150.40_ECG_signal\rpeaks_fig\non_mace/' ##
elif mode == 1:
    file_path = [os.path.join(fc500_file_path, 'mace'), os.path.join(fc150_file_path, 'mace'), os.path.join(fc40_file_path, 'mace')]
    save_path = r'V:\dunwei\MACE\dataset\ECG_signal\ch1\sr10000_0.5_500.150.40_ECG_signal\rpeaks_fig\mace/' ##

In [ ]:
if not os.path.exists(save_path):
    os.makedirs(save_path)
    print(f"Directory {save_path} created.")
else:
    print(f"Directory {save_path} already exists.")

In [ ]:
research_id = process_research_id(file_path)
random_idx = np.random.choice(len(research_id), 90, replace=False)
random_idx = np.sort(random_idx)
select_research_id = [research_id[i] for i in random_idx]
print(select_research_id)

In [ ]:
for idx, id in enumerate(tqdm(select_research_id, desc='Processing ECG plot')):

    ECG_segment_data_list = []
    r_peaks_info_list = []
    for i, path in enumerate(file_path):
        # resample_rate = resample_rates[i]
        file_name = os.path.join(path, id + '.npy')
        # if mode == 0:
        #     file_name = os.path.join(path, f'{non_mace_test_id}.npy')
        # elif mode == 1:
        #     file_name = os.path.join(path, f'{mace_test_id}.npy')
        ECG_segment_data = np.load(file_name, allow_pickle=True)
        rpeaks, info = nk.ecg_peaks(ECG_segment_data, sampling_rate=resample_rate, correct_artifacts=True, method="neurokit")
        info_cleaned, peaks_cleaned = nk.signal_fixpeaks(info["ECG_R_Peaks"], sampling_rate=resample_rate, method="Kubios", iterative=True)
        info['ECG_R_Peaks'] = peaks_cleaned
        ECG_segment_data_list.append(ECG_segment_data)
        r_peaks_info_list.append(info)

    fig, ax = plt.subplots(3, 1, figsize=(12, 10))
    for j in range(len(ECG_segment_data_list)):
        # resample_rate = resample_rates[j]
        rpeaks_count = len(r_peaks_info_list[j]['ECG_R_Peaks'])
        time = np.linspace(0, len(ECG_segment_data_list[j]) / resample_rate, len(ECG_segment_data_list[j]))
        ax[j].plot(time, ECG_segment_data_list[j], color='Tab:blue', label='ECG Signal')
        if mode == 0:
            ax[j].set_title(f'Non {id} ECG with 0.5 ~ {freq_cut[j]} Hz / resample:{resample_rate} / Rpeaks count:{rpeaks_count}', fontsize=10, fontweight='bold')
            # ax[j].set_title(f'Non {non_mace_test_id} ECG with 0.5 ~ {freq_cut[j]} Hz / resample:{resample_rate} / Rpeaks count:{rpeaks_count}', fontsize=10, fontweight='bold')
        elif mode == 1:
            ax[j].set_title(f'{id} ECG with 0.5 ~ {freq_cut[j]} Hz / resample:{resample_rate} / Rpeaks count:{rpeaks_count}', fontsize=10, fontweight='bold')
            # ax[j].set_title(f'{mace_test_id} ECG with 0.5 ~ {freq_cut[j]} Hz / resample:{resample_rate} / Rpeaks count:{rpeaks_count}', fontsize=10, fontweight='bold')
        ax[j].set_xlabel('Time (s)')
        ax[j].set_ylabel('Voltage (mV)')
        ax[j].scatter(r_peaks_info_list[j]['ECG_R_Peaks'] / resample_rate, ECG_segment_data_list[j][r_peaks_info_list[j]['ECG_R_Peaks']], color='red', label='R Peaks', marker='o')
        # ax[j].set_xlim(0, 20) ##
        ax[j].legend()
        ax[j].grid(True, linestyle='-', alpha=0.5)
    plt.tight_layout()
    if not os.path.exists(os.path.join(save_path, f'{select_research_id[idx]}')):
        os.makedirs(os.path.join(save_path, f'{select_research_id[idx]}'))
        # print(f"Directory {os.path.join(save_path, f'{select_research_id[idx]}')} created.")
    plt.savefig(os.path.join(save_path, f'{select_research_id[idx]}', f'{select_research_id[idx]}_ECG_R_peaks_{save_time}.png'), dpi=600, bbox_inches='tight')
    plt.close(fig)